In [ ]:
from pathlib import Path
import random
import pandas as pd
import cv2

from ultralytics import YOLO

random.seed(42)

In [ ]:
root_folder = Path("slide_data_path")

'''
Data should be broken up into the format:

slide_data_path/
    slide_1/
        tile1.png
        tile2.png
        tile3.png
        ...
    slide_2/
        tile1.png
        tile2.png
        tile3.png
    ...
'''
model_path = "yolo_model_weights.pt"

num_regions_per_slide = 10

blast_class_id = 1
normal_class_id = 2

In [ ]:
model = YOLO(model_path)
print(model.names)

In [ ]:
slide_folders = sorted(
    [p for p in root_folder.iterdir() if p.is_dir()],
    key=lambda p: int(p.name)
)
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

In [ ]:
viz_root = Path("yolo_box_visuals")
viz_root.mkdir(exist_ok=True)

results_rows = []

for slide_folder in slide_folders:
    region_folder = slide_folder / "focus_regions" / "high_mag_unannotated"

    if not region_folder.exists():
        print(f"\nSlide: {slide_folder.name}")
        print("  Folder not found, skipping.")
        continue

    image_files = [
        p for p in region_folder.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    if len(image_files) == 0:
        print(f"\nSlide: {slide_folder.name}")
        print("  No image files found, skipping.")
        continue

    n_to_sample = min(num_regions_per_slide, len(image_files))
    sampled_files = random.sample(image_files, n_to_sample)

    slide_viz_folder = viz_root / slide_folder.name
    slide_viz_folder.mkdir(exist_ok=True)

    blast_count = 0
    normal_count = 0

    for img_path in sampled_files:
        results = model(str(img_path), verbose=False)

        for result in results:
            if result.boxes is not None and result.boxes.cls is not None:
                detected_classes = result.boxes.cls.cpu().numpy().astype(int)
                blast_count += (detected_classes == blast_class_id).sum()
                normal_count += (detected_classes == normal_class_id).sum()

            plotted = result.plot()
            save_path = slide_viz_folder / img_path.name
            cv2.imwrite(str(save_path), plotted)

    denom = blast_count + normal_count
    blast_ratio = blast_count / denom if denom > 0 else 0

    print(f"\nSlide: {slide_folder.name}")
    print(f"  Sampled regions: {n_to_sample}")
    print(f"  Blast count: {blast_count}")
    print(f"  Normal count: {normal_count}")
    print(f"  Blast ratio: {blast_ratio:.4f}")

    results_rows.append({
        "slide": slide_folder.name,
        "sampled_regions": n_to_sample,
        "blast_count": int(blast_count),
        "normal_count": int(normal_count),
        "blast_ratio": float(blast_ratio)
    })

df = pd.DataFrame(results_rows)
df

In [ ]:
save_name = "saved_predictions.csv"
df.to_csv(save_name, index=False)
print(f"Saved to {save_name}")

In [ ]:
# code to create box plot

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Arial"

normal_csv = "normal_saved_predictions.csv"
aml_csv = "AML_saved_predictions.csv"

In [ ]:
normal_df = pd.read_csv(normal_csv)
aml_df = pd.read_csv(aml_csv)

print(normal_df.head())
print(aml_df.head())

normal_blast_pct = normal_df["blast_ratio"] * 100
aml_blast_pct = aml_df["blast_ratio"] * 100

print("Normal n =", len(normal_blast_pct))
print("AML n =", len(aml_blast_pct))

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 4.0), dpi=150)

bp = ax.boxplot(
    [normal_blast_pct, aml_blast_pct],
    patch_artist=True,
    widths=0.6,
    showfliers=False
)

box_colors = ["#4C90C0", "#F2545B"]

for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_edgecolor(color)
    patch.set_linewidth(1.5)

for whisker, color in zip(
    bp["whiskers"],
    [box_colors[0], box_colors[0], box_colors[1], box_colors[1]]
):
    whisker.set_color(color)
    whisker.set_linewidth(1.0)

for cap, color in zip(
    bp["caps"],
    [box_colors[0], box_colors[0], box_colors[1], box_colors[1]]
):
    cap.set_color(color)
    cap.set_linewidth(1.0)

for median, color in zip(bp["medians"], box_colors):
    median.set_color(color)
    median.set_linewidth(2)

ax.set_xticks([1, 2])
ax.set_xticklabels(["Normal", "AML"], fontsize=22)
ax.set_ylabel("ALLocate Blast %", fontsize=26, fontweight="bold")

ax.set_ylim(0, 100)
ax.tick_params(axis="y", labelsize=18, length=0)
ax.tick_params(axis="x", length=0)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#444444")
ax.spines["bottom"].set_color("#444444")

plt.tight_layout()
plt.show()

In [ ]:
fig.savefig("blast_ratio_boxplot.png", dpi=300, bbox_inches="tight")